In [4]:
import pickle
import gzip
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.model_selection import KFold, GridSearchCV
from sklearn.metrics import mean_squared_error

# Load preprocessed DeepSynergy data
with gzip.open("data_test_fold0_tanh.p.gz", "rb") as f:
    X_tr, X_val, X_train, X_test, y_tr, y_val, y_train, y_test = pickle.load(f)

# Combine into a full dataset for CV
X = np.concatenate([X_train, X_test], axis=0)
y = np.concatenate([y_train, y_test], axis=0)

# Define hyperparameter grids based on DeepSynergy supplementary material
param_grid_rf = {
    'n_estimators': [128, 512, 1024, 2048],
    'max_features': ['sqrt', 256]
}

param_grid_ridge = {
    'alpha': [0.1, 1.0, 10.0, 100.0]
}

# Setup Nested CV
outer_cv = KFold(n_splits=5, shuffle=True, random_state=42)
inner_cv = KFold(n_splits=3, shuffle=True, random_state=1)

rf_mse_scores = []
ridge_mse_scores = []

for train_ix, test_ix in outer_cv.split(X):
    X_train, X_test = X[train_ix], X[test_ix]
    y_train, y_test = y[train_ix], y[test_ix]

    # --- Random Forest ---
    rf = RandomForestRegressor(random_state=0)
    rf_grid = GridSearchCV(rf, param_grid_rf, cv=inner_cv, scoring='neg_mean_squared_error', n_jobs=-1)
    rf_grid.fit(X_train, y_train)
    best_rf = rf_grid.best_estimator_
    y_pred_rf = best_rf.predict(X_test)
    rf_mse = mean_squared_error(y_test, y_pred_rf)
    rf_mse_scores.append(rf_mse)
    print(f"Random Forest Fold MSE: {rf_mse:.2f} | Best Params: {rf_grid.best_params_}")

    # --- Ridge Regression ---
    ridge = Ridge()
    ridge_grid = GridSearchCV(ridge, param_grid_ridge, cv=inner_cv, scoring='neg_mean_squared_error', n_jobs=-1)
    ridge_grid.fit(X_train, y_train)
    best_ridge = ridge_grid.best_estimator_
    y_pred_ridge = best_ridge.predict(X_test)
    ridge_mse = mean_squared_error(y_test, y_pred_ridge)
    ridge_mse_scores.append(ridge_mse)
    print(f"Ridge Regression Fold MSE: {ridge_mse:.2f} | Best alpha: {ridge_grid.best_params_['alpha']}")

# Final evaluation
print("\nFinal Evaluation:")
print(f"Random Forest MSEs: {rf_mse_scores}")
print(f"Mean Random Forest MSE: {np.mean(rf_mse_scores):.2f}")

print(f"Ridge Regression MSEs: {ridge_mse_scores}")
print(f"Mean Ridge Regression MSE: {np.mean(ridge_mse_scores):.2f}")


ModuleNotFoundError: No module named 'numpy'